In [1]:
# ============================================================
# MERGING 36 CLEAN JSON DATASETS -> 2 CONSOLIDATED FILES
# (profiles_all.json, jobs_all.json) + addition of "type" field
# ============================================================

import json
import glob
import os
import re
import pandas as pd


from google.colab import files
uploaded = files.upload()
DATA_DIR = "/content"

all_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.json")))
print(f"Found {len(all_files)} files {DATA_DIR}:")
for f in all_files:
    print(" -", os.path.basename(f))

FILENAME_PATTERN = re.compile(
    r"^(jobs|profiles)_([a-z_]+)_(green|lignite)_clean$", re.IGNORECASE
)

def parse_filename(path):
    name = os.path.splitext(os.path.basename(path))[0]
    m = FILENAME_PATTERN.match(name)
    if not m:
        print(f"  [WARNING] File name not recognized: {name}")
        return None
    entity_type, country, category = m.groups()
    return entity_type.lower(), country.lower(), category.lower()


profiles_records = []
jobs_records = []
skipped_empty = []
unmatched_files = []

for path in all_files:
    parsed = parse_filename(path)
    if parsed is None:
        unmatched_files.append(os.path.basename(path))
        continue
    entity_type, country, category = parsed

    with open(path, "r", encoding="utf-8") as f:
        try:
            records = json.load(f)
        except json.JSONDecodeError as e:
            print(f"  [Read error] {path}: {e}")
            continue

    if not isinstance(records, list):
        records = [records]

    if len(records) == 0:
        skipped_empty.append(os.path.basename(path))
        continue

    for r in records:
        r["type"] = category                       # "green" or "lignite"
        r.setdefault("country", country)            # fills in the country ONLY if it does not already exist (e.g., in jobs)

    if entity_type == "profiles":
        profiles_records.extend(records)
    elif entity_type == "jobs":
        jobs_records.extend(records)

print(f"\nIgnored due to unknown name: {unmatched_files}")
print(f"Empty files (0 records): {skipped_empty}")
print(f"Total PROFILES records: {len(profiles_records)}")
print(f"Total JOBS records: {len(jobs_records)}")

# ---- ID Uniqueness Check ----
def check_duplicates(records, name):
    ids = pd.Series([r.get("id") for r in records])
    n_total, n_missing = len(ids), ids.isna().sum()
    n_unique = ids.nunique()
    print(f"\n[{name}] Records: {n_total} | Unique id: {n_unique} | Without id: {n_missing}")
    if n_unique < n_total - n_missing:
        dupes = ids[ids.duplicated(keep=False)].dropna().unique()
        print(f"  [Warning] {n_total - n_unique - n_missing} duplicates id! {list(dupes[:10])}")
    else:
        print("  All IDs are unique ")

check_duplicates(profiles_records, "PROFILES")
check_duplicates(jobs_records, "JOBS")


OUTPUT_DIR = DATA_DIR

profiles_out = os.path.join(OUTPUT_DIR, "profiles_all.json")
jobs_out = os.path.join(OUTPUT_DIR, "jobs_all.json")

with open(profiles_out, "w", encoding="utf-8") as f:
    json.dump(profiles_records, f, ensure_ascii=False, indent=2)
with open(jobs_out, "w", encoding="utf-8") as f:
    json.dump(jobs_records, f, ensure_ascii=False, indent=2)

print(f"\nSaved: {profiles_out}  ({len(profiles_records)} records)")
print(f"Saved: {jobs_out}  ({len(jobs_records)} records)")

print("\n--- Profiles per country & type ---")
print(pd.DataFrame(profiles_records).groupby(["country", "type"]).size().unstack(fill_value=0))

print("\n--- Jobs per country & type ---")
print(pd.DataFrame(jobs_records).groupby(["country", "type"]).size().unstack(fill_value=0))

Found 35 files /content:
 - jobs_czechia_green_clean.json
 - jobs_czechia_lignite_clean.json
 - jobs_france_green_clean.json
 - jobs_france_lignite_clean.json
 - jobs_germany_green_clean.json
 - jobs_germany_lignite_clean.json
 - jobs_greece_green_clean.json
 - jobs_greece_lignite_clean.json
 - jobs_italy_green_clean.json
 - jobs_italy_lignite_clean.json
 - jobs_poland_green_clean.json
 - jobs_poland_lignite_clean.json
 - jobs_romania_green_clean.json
 - jobs_spain_green_clean.json
 - jobs_spain_lignite_clean.json
 - jobs_sweden_green_clean.json
 - jobs_sweden_lignite_clean.json
 - profiles_czechia_green_clean.json
 - profiles_czechia_lignite_clean.json
 - profiles_france_green_clean.json
 - profiles_france_lignite_clean.json
 - profiles_germany_green_clean.json
 - profiles_germany_lignite_clean.json
 - profiles_greece_green_clean.json
 - profiles_greece_lignite_clean.json
 - profiles_italy_green_clean.json
 - profiles_italy_lignite_clean.json
 - profiles_poland_green_clean.json
 - pro

In [2]:
import pandas as pd

print("--- PROFILES (Top 5) ---")
df_profiles = pd.read_json('profiles_all.json')
display(df_profiles.head(5))

print("\n--- JOBS (Top 5) ---")
df_jobs = pd.read_json('jobs_all.json')
display(df_jobs.head(5))

--- PROFILES (Top 5) ---


,skills,sectors,id,full_name,location,content,occupation,occupation_uris,city,company,...,startdate,university_name,university_raw,university_country,university_location,ultimate_parent_school_name,url,source,source_id,type
0,[],"[Architectural activities, Interior design act...",130666,Petr Smolny,Czech Republic,None,"Všechno jde, když se chce.",[http://data.europa.eu/esco/occupation/efa8163...,Podborany,Ceská Pošta SP,...,None,None,None,None,None,None,linkedin.com/in/petr-smolny-aa26704,revelio,273668659,green
1,[],"[CONSTRUCTION, Engineering activities and rela...",134942,Vojtěch Mahdal,Czech Republic,None,"Projekce FVE, distribučních a energetických sy...",[http://data.europa.eu/esco/occupation/d7d986e...,Ostrava,CEZ as,...,2016-01-01,VŠB - Technical University of Ostrava,VSB - Technical University of Ostrava,Czech Republic,None,VŠB - Technical University of Ostrava,linkedin.com/in/vojt%C4%9Bch-mahdal-354aa1257,revelio,1036813681,green
2,[],"[Architectural activities, Interior design act...",143872,Andrea Pašková,Czech Republic,None,Návrhář interiérů ve společnosti DDAANN archit...,[http://data.europa.eu/esco/occupation/efa8163...,empty,DDAANN architects,...,None,Technical University in Zvolen,Technická univerzita vo Zvolene,Slovakia,None,Technical University in Zvolen,linkedin.com/in/andrea-pa%C5%A1kov%C3%A1-900b9...,revelio,2274714060,green
3,[],"[Architectural activities, Interior design act...",145204,marek grabovsky,Czech Republic,None,Intern at S.A.B. International,[http://data.europa.eu/esco/occupation/efa8163...,empty,S.A.B. International,...,None,None,None,None,None,None,linkedin.com/in/marek-grabovsky-3941662b,revelio,601611697,green
4,[http://data.europa.eu/esco/skill/7ce196c4-65c...,[Engineering activities and related technical ...,149707,Daniel Frídel,Czech Republic,Hi there! ✌🏻 Welcome to my profile. To sum it ...,Life-long disciple of life,[http://data.europa.eu/esco/occupation/ac1fc6a...,Prague,Vodafone Group Plc,...,2016-01-01,University of New York in Prague,University of New York in Prague,Czech Republic,None,University of New York in Prague,linkedin.com/in/daniel-fr%C3%ADdel-553b84121,revelio,973397312,green



--- JOBS (Top 5) ---


,organization,skills,occupations,sectors,id,title,description,experience_level,type,location,location_code,nuts1,nuts2,nuts3,upload_date,source,source_id,country
0,"ENVISTONE, spol. s r.o.",[http://data.europa.eu/esco/skill/1fe682ad-b8a...,[http://data.europa.eu/esco/occupation/fd6a1c1...,"[Collection of non-hazardous waste, Other wast...",6247696,OPERATING WASTE PROCESSING AND RECYCLING EQUIP...,CZ-ISCO: 81892 WASTE PROCESSING AND RECYCLING ...,None,green,None,CZ,None,None,None,2026-01-01,eures-escox,MjE2NzU5NDA3ODAgNg,czechia
1,DAS Czech Republic s.r.o.,[http://data.europa.eu/esco/skill/09e7cd13-ea3...,[http://data.europa.eu/esco/occupation/109e0a5...,[Activities of head offices],6247730,Managers in the field of quality and certifica...,Entry possible IMMEDIATELY. Job description: R...,None,green,"Hranice, CZ",CZ,Česko,Střední Morava,Olomoucký kraj,2025-10-16,eures-escox,MjI0NTMyMjA3OTIgNg,czechia
2,KOM invest s.r.o.,[http://data.europa.eu/esco/skill/199f7919-511...,[http://data.europa.eu/esco/isco/C8121],[],6251676,"Operation of metal processing equipment, Opera...","We require: an extract from RT, appropriate he...",None,green,None,CZ,Česko,Střední Čechy,Středočeský kraj,2026-01-06,eures-escox,MTkwODc1MjA3MDkgNg,czechia
3,Kaufland Česká republika v.o.s.,[http://data.europa.eu/esco/skill/0450eaee-1a0...,[http://data.europa.eu/esco/isco/C9112],[],6252264,"Cleaning and disposal workers, Cleaners of pro...",More details: Ensures cleanliness and order in...,None,green,None,CZ,Česko,Střední Čechy,Středočeský kraj,2026-01-08,eures-escox,MjU5MjcwODA3MjUgNg,czechia
4,Daimler Buses Česká republika s.r.o.,[http://data.europa.eu/esco/skill/012d6406-016...,[http://data.europa.eu/esco/occupation/4ad4024...,[Repair and maintenance of motor vehicles],6254217,"Bus and trolleybus mechanic and repairman, Bus...",Job description: - Detecting the causes of mac...,None,green,"Holýšov, CZ",CZ,Česko,Jihozápad,Plzeňský kraj,2025-10-07,eures-escox,Mjc5ODA0NjA3MjcgNg,czechia


In [3]:
import pandas as pd

def check_missing_skills(file_path):
    print(f"--- Checking file: {file_path} ---")

    # Load JSON file (add lines=True if needed)
    df = pd.read_json(file_path)
    total_records = len(df)
    print(f"Total records: {total_records}")

    if 'skills' not in df.columns:
        print("WARNING: Column 'skills' was not found!\n")
        return

    # Helper function safely checking for empty lists or None
    def is_empty_list(val):
        if val is None:
            return True
        if isinstance(val, (list, tuple)):
            return len(val) == 0
        return False

    # Count empty lists
    empty_list_mask = df['skills'].apply(is_empty_list)
    empty_count = empty_list_mask.sum()
    percentage = (empty_count / total_records) * 100

    print(f"Empty skills lists ([]): {empty_count} ({percentage:.2f}%)")
    print(f"Valid populated skills: {total_records - empty_count} ({(100 - percentage):.2f}%)\n")

# Run check on both datasets
check_missing_skills('profiles_all.json')
check_missing_skills('jobs_all.json')

--- Checking file: profiles_all.json ---
Total records: 334151
Empty skills lists ([]): 303400 (90.80%)
Valid populated skills: 30751 (9.20%)

--- Checking file: jobs_all.json ---
Total records: 219809
Empty skills lists ([]): 13159 (5.99%)
Valid populated skills: 206650 (94.01%)



In [5]:
# ============================================================
# FILL MISSING "skills" VALUES IN profiles_all.json USING
# ESCO occupationSkillRelations_en.csv (essential skills only)
# ============================================================

import json
import pandas as pd

# ---- 1. PATHS (edit these to match your Colab/Drive setup) ----
PROFILES_PATH = "/content/profiles_all.json"        # your merged profiles file
ESCO_RELATIONS_PATH = "/content/occupationSkillRelations_en.csv"      # the uploaded ESCO file
OUTPUT_PATH = "/content/profiles_all_filled.json"

# ---- 2. LOAD THE ESCO OCCUPATION -> SKILL RELATIONS FILE ----
relations_df = pd.read_csv(
    ESCO_RELATIONS_PATH,
    usecols=["occupationUri", "relationType", "skillUri"]
)

# Keep ONLY essential relations, as requested
essential_relations = relations_df[relations_df["relationType"] == "essential"]

# Build a lookup dict: occupationUri -> list of unique essential skillUris
occ_to_essential_skills = (
    essential_relations.groupby("occupationUri")["skillUri"]
    .apply(lambda s: sorted(set(s)))   # sorted(set(...)) removes accidental duplicates
    .to_dict()
)

print(f"Loaded {len(occ_to_essential_skills)} occupations with essential skills mapped.")

# ---- 3. LOAD THE MERGED PROFILES DATASET ----
with open(PROFILES_PATH, "r", encoding="utf-8") as f:
    profiles = json.load(f)

print(f"Loaded {len(profiles)} profile records.")

# ---- 4. HELPER: IS THE "skills" FIELD ACTUALLY MISSING/EMPTY? ----
def is_missing_skills(skills_value):
    # Covers: key absent (None), NaN slipped in as float, empty list, empty string
    if skills_value is None:
        return True
    if isinstance(skills_value, float):
        return True
    if isinstance(skills_value, list) and len(skills_value) == 0:
        return True
    if isinstance(skills_value, str) and skills_value.strip() == "":
        return True
    return False

# ---- 5. HELPER: GET ESSENTIAL SKILLS FOR A LIST OF occupation_uris ----
def get_essential_skills_for_occupations(occupation_uris, occ_to_essential):
    if not occupation_uris:
        return []
    resolved = set()
    for uri in occupation_uris:
        # The ESCO relations file only covers real occupation URIs
        # (".../esco/occupation/..."), not ISCO group codes
        # (".../esco/isco/C7411") -> those are simply skipped here
        if "/occupation/" not in uri:
            continue
        resolved.update(occ_to_essential.get(uri, []))
    return sorted(resolved)

# ---- 6. FILL THE MISSING "skills" VALUES ----
filled_count = 0
still_missing_count = 0
already_present_count = 0

for record in profiles:
    current_skills = record.get("skills")

    if not is_missing_skills(current_skills):
        record["skills_source"] = "original"
        already_present_count += 1
        continue

    occupation_uris = record.get("occupation_uris", [])
    new_skills = get_essential_skills_for_occupations(occupation_uris, occ_to_essential_skills)

    if new_skills:
        record["skills"] = new_skills
        record["skills_source"] = "esco_essential"
        filled_count += 1
    else:
        record["skills"] = []           # explicitly empty, not a missing key
        record["skills_source"] = "unresolved"
        still_missing_count += 1

# ---- 7. SUMMARY ----
total = len(profiles)
print(f"\nTotal profiles: {total}")
print(f"Already had skills (untouched): {already_present_count} ({already_present_count/total:.1%})")
print(f"Filled from ESCO essential skills: {filled_count} ({filled_count/total:.1%})")
print(f"Still unresolved (no essential-skill match found): {still_missing_count} ({still_missing_count/total:.1%})")

# ---- 8. BREAKDOWN BY COUNTRY AND TYPE (green vs lignite) ----
summary_df = pd.DataFrame(profiles)[["country", "type", "skills_source"]]
print("\nBreakdown by country and type:")
print(summary_df.groupby(["country", "type", "skills_source"]).size().unstack(fill_value=0))

# ---- 9. SAVE THE UPDATED DATASET ----
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(profiles, f, ensure_ascii=False, indent=2)

print(f"\nSaved filled dataset to: {OUTPUT_PATH}")

Loaded 3039 occupations with essential skills mapped.
Loaded 334151 profile records.

Total profiles: 334151
Already had skills (untouched): 30751 (9.2%)
Filled from ESCO essential skills: 303400 (90.8%)
Still unresolved (no essential-skill match found): 0 (0.0%)

Breakdown by country and type:
skills_source           esco_essential  original
country        type                             
Czech Republic green              2736       328
               lignite            2684       352
France         green             50936      3433
               lignite           42844      2389
Germany        green             31341      3216
               lignite           23187      2520
Greece         green              3840      1073
               lignite            2546       701
Italy          green             33892      3428
               lignite           21225      2065
Poland         green              7625      1119
               lignite            7698      1033
Romania        gre

In [6]:
# ============================================================
# QUICK INSPECTION OF profiles_all_filled.json
# 1) Sample of first 5 records
# 2) Missing-values check on the "skills" field
# 3) Mean and median number of skills per profile
# ============================================================

import json
import pandas as pd

FILLED_PROFILES_PATH = "/content/profiles_all_filled.json"  # adjust path

with open(FILLED_PROFILES_PATH, "r", encoding="utf-8") as f:
    profiles = json.load(f)

df = pd.DataFrame(profiles)
print(f"Total records loaded: {len(df)}")

# ---- 1. SHOW THE FIRST 5 RECORDS ----
print("\n" + "=" * 60)
print("SAMPLE: first 5 records")
print("=" * 60)
for i, record in enumerate(profiles[:5]):
    print(f"\n--- Record {i+1} ---")
    print(json.dumps(record, ensure_ascii=False, indent=2))

# ---- 2. MISSING VALUES CHECK ON "skills" ----
# We check TWO things separately, because they mean different stages of the pipeline:
#   (a) truly missing key / None            -> would mean something went wrong in the fill step
#   (b) present but empty list []           -> means "unresolved": no essential-skill match was found
def skills_status(skills_value):
    if skills_value is None:
        return "missing_none"
    if isinstance(skills_value, float):  # NaN edge case
        return "missing_nan"
    if isinstance(skills_value, list) and len(skills_value) == 0:
        return "empty_list"
    return "has_skills"

df["skills_status"] = df["skills"].apply(skills_status)

print("\n" + "=" * 60)
print("MISSING VALUES CHECK on 'skills'")
print("=" * 60)
status_counts = df["skills_status"].value_counts()
status_pct = (status_counts / len(df) * 100).round(2)
summary = pd.DataFrame({"count": status_counts, "percent": status_pct})
print(summary)

if "skills_source" in df.columns:
    print("\nCross-check against skills_source (from the fill step):")
    print(df.groupby(["skills_source", "skills_status"]).size().unstack(fill_value=0))

# ---- 3. MEAN AND MEDIAN NUMBER OF SKILLS PER PROFILE ----
df["n_skills"] = df["skills"].apply(lambda s: len(s) if isinstance(s, list) else 0)

print("\n" + "=" * 60)
print("SKILL COUNT STATISTICS")
print("=" * 60)
print(f"Mean number of skills per profile (all records):   {df['n_skills'].mean():.2f}")
print(f"Median number of skills per profile (all records): {df['n_skills'].median():.2f}")

# Same stats but excluding "unresolved" (empty-list) records,
# to see the real distribution among profiles that actually got skills
resolved_df = df[df["n_skills"] > 0]
print(f"\nAmong profiles WITH at least one skill (n={len(resolved_df)}):")
print(f"Mean number of skills per profile:   {resolved_df['n_skills'].mean():.2f}")
print(f"Median number of skills per profile: {resolved_df['n_skills'].median():.2f}")

# Optional: breakdown by country and type, useful to spot problem countries
print("\nMean skill count by country and type:")
print(df.groupby(["country", "type"])["n_skills"].mean().round(2).unstack(fill_value=0))

Total records loaded: 334151

SAMPLE: first 5 records

--- Record 1 ---
{
  "skills": [
    "http://data.europa.eu/esco/skill/0332f526-6dc0-4f09-8a7c-2b9473c50736",
    "http://data.europa.eu/esco/skill/0da516ee-e70e-4384-be13-f5ff80be8127",
    "http://data.europa.eu/esco/skill/2c71ffc8-dae9-4584-ae7f-5b3a78c8cc69",
    "http://data.europa.eu/esco/skill/3a0d4d83-6392-4a81-87f1-5b57b08110b5",
    "http://data.europa.eu/esco/skill/44d5340c-45df-4478-9e55-2363342a7324",
    "http://data.europa.eu/esco/skill/4c58528e-bdaa-43ad-8f5a-8ad0b8cd4bbb",
    "http://data.europa.eu/esco/skill/4ccf7d02-764c-4129-a1df-fa1dd2e12647",
    "http://data.europa.eu/esco/skill/4f1559a5-c0e6-4950-a66c-c57e12380602",
    "http://data.europa.eu/esco/skill/59ea80e1-463a-4dba-82c6-d0b6d577d532",
    "http://data.europa.eu/esco/skill/68698869-c13c-4563-adc7-118b7644f45d",
    "http://data.europa.eu/esco/skill/6db3ac8e-8636-4ced-965b-75e35c3193c3",
    "http://data.europa.eu/esco/skill/7111b95d-0ce3-441a-9d92-4c7

In [7]:
import json
import pandas as pd

with open("/content/jobs_all.json", encoding="utf-8") as f:
    jobs = json.load(f)

print(pd.DataFrame(jobs)["type"].value_counts())

type
lignite    115161
green      104648
Name: count, dtype: int64


In [8]:
# ============================================================
# FILL MISSING "skills" VALUES IN jobs_all.json USING
# ESCO occupationSkillRelations_en.csv (essential skills only)
# ============================================================

import json
import pandas as pd

# ---- 1. PATHS (edit these to match your Colab/Drive setup) ----
JOBS_PATH = "/content/jobs_all.json"                # your merged jobs file
ESCO_RELATIONS_PATH = "/content/occupationSkillRelations_en.csv"      # the uploaded ESCO file
OUTPUT_PATH = "/content/jobs_all_filled.json"

# ---- 2. LOAD THE ESCO OCCUPATION -> SKILL RELATIONS FILE ----
relations_df = pd.read_csv(
    ESCO_RELATIONS_PATH,
    usecols=["occupationUri", "relationType", "skillUri"]
)

# Keep ONLY essential relations, as requested
essential_relations = relations_df[relations_df["relationType"] == "essential"]

# Build a lookup dict: occupationUri -> list of unique essential skillUris
occ_to_essential_skills = (
    essential_relations.groupby("occupationUri")["skillUri"]
    .apply(lambda s: sorted(set(s)))
    .to_dict()
)

print(f"Loaded {len(occ_to_essential_skills)} occupations with essential skills mapped.")

# ---- 3. LOAD THE MERGED JOBS DATASET ----
with open(JOBS_PATH, "r", encoding="utf-8") as f:
    jobs = json.load(f)

print(f"Loaded {len(jobs)} job records.")

# ---- 4. HELPER: IS THE "skills" FIELD ACTUALLY MISSING/EMPTY? ----
def is_missing_skills(skills_value):
    if skills_value is None:
        return True
    if isinstance(skills_value, float):  # NaN edge case
        return True
    if isinstance(skills_value, list) and len(skills_value) == 0:
        return True
    if isinstance(skills_value, str) and skills_value.strip() == "":
        return True
    return False

# ---- 5. HELPER: GET ESSENTIAL SKILLS FOR A LIST OF occupation URIs ----
def get_essential_skills_for_occupations(occupation_uris, occ_to_essential):
    if not occupation_uris:
        return []
    resolved = set()
    for uri in occupation_uris:
        # The ESCO relations file only covers real occupation URIs
        # (".../esco/occupation/..."), not ISCO group codes
        # (".../esco/isco/C2433") -> those are simply skipped here
        if "/occupation/" not in uri:
            continue
        resolved.update(occ_to_essential.get(uri, []))
    return sorted(resolved)

# ---- 6. FILL THE MISSING "skills" VALUES ----
filled_count = 0
still_missing_count = 0
already_present_count = 0

for record in jobs:
    current_skills = record.get("skills")

    if not is_missing_skills(current_skills):
        record["skills_source"] = "original"
        already_present_count += 1
        continue

    # NOTE: the field is called "occupations" in the jobs schema
    # (it was "occupation_uris" in the profiles schema)
    occupation_uris = record.get("occupations", [])
    new_skills = get_essential_skills_for_occupations(occupation_uris, occ_to_essential_skills)

    if new_skills:
        record["skills"] = new_skills
        record["skills_source"] = "esco_essential"
        filled_count += 1
    else:
        record["skills"] = []
        record["skills_source"] = "unresolved"
        still_missing_count += 1

# ---- 7. SUMMARY ----
total = len(jobs)
print(f"\nTotal jobs: {total}")
print(f"Already had skills (untouched): {already_present_count} ({already_present_count/total:.1%})")
print(f"Filled from ESCO essential skills: {filled_count} ({filled_count/total:.1%})")
print(f"Still unresolved (no essential-skill match found): {still_missing_count} ({still_missing_count/total:.1%})")

# ---- 8. BREAKDOWN BY COUNTRY AND TYPE (green vs lignite) ----
summary_df = pd.DataFrame(jobs)[["country", "type", "skills_source"]]
print("\nBreakdown by country and type:")
print(summary_df.groupby(["country", "type", "skills_source"]).size().unstack(fill_value=0))

# ---- 9. SAVE THE UPDATED DATASET ----
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(jobs, f, ensure_ascii=False, indent=2)

print(f"\nSaved filled dataset to: {OUTPUT_PATH}")

Loaded 3039 occupations with essential skills mapped.
Loaded 219809 job records.

Total jobs: 219809
Already had skills (untouched): 206650 (94.0%)
Filled from ESCO essential skills: 13159 (6.0%)
Still unresolved (no essential-skill match found): 0 (0.0%)

Breakdown by country and type:
skills_source    esco_essential  original
country type                             
czechia green                 0       371
        lignite              41       196
france  green              3545     37157
        lignite            4302     33852
germany green              1078     22711
        lignite            2207     45802
greece  green               102      3522
        lignite              76      2811
italy   green                28       218
        lignite              34       480
poland  green               103       442
        lignite             120       271
romania green                 0        76
spain   green               362       536
        lignite             606       35

In [9]:
# ============================================================
# QUICK INSPECTION OF jobs_all_filled.json
# 1) Sample of first 5 records
# 2) Missing-values check on the "skills" field
# 3) Mean and median number of skills per job record
# ============================================================

import json
import pandas as pd

FILLED_JOBS_PATH = "/content/jobs_all_filled.json"

with open(FILLED_JOBS_PATH, "r", encoding="utf-8") as f:
    jobs = json.load(f)

df = pd.DataFrame(jobs)
print(f"Total records loaded: {len(df)}")

# ---- 1. SHOW THE FIRST 5 RECORDS ----
print("\n" + "=" * 60)
print("SAMPLE: first 5 records")
print("=" * 60)
for i, record in enumerate(jobs[:5]):
    print(f"\n--- Record {i+1} ---")
    print(json.dumps(record, ensure_ascii=False, indent=2))

# ---- 2. MISSING VALUES CHECK ON "skills" ----
def skills_status(skills_value):
    if skills_value is None:
        return "missing_none"
    if isinstance(skills_value, float):  # NaN edge case
        return "missing_nan"
    if isinstance(skills_value, list) and len(skills_value) == 0:
        return "empty_list"
    return "has_skills"

df["skills_status"] = df["skills"].apply(skills_status)

print("\n" + "=" * 60)
print("MISSING VALUES CHECK on 'skills'")
print("=" * 60)
status_counts = df["skills_status"].value_counts()
status_pct = (status_counts / len(df) * 100).round(2)
summary = pd.DataFrame({"count": status_counts, "percent": status_pct})
print(summary)

if "skills_source" in df.columns:
    print("\nCross-check against skills_source (from the fill step):")
    print(df.groupby(["skills_source", "skills_status"]).size().unstack(fill_value=0))

# ---- 3. MEAN AND MEDIAN NUMBER OF SKILLS PER JOB ----
df["n_skills"] = df["skills"].apply(lambda s: len(s) if isinstance(s, list) else 0)

print("\n" + "=" * 60)
print("SKILL COUNT STATISTICS")
print("=" * 60)
print(f"Mean number of skills per job (all records):   {df['n_skills'].mean():.2f}")
print(f"Median number of skills per job (all records): {df['n_skills'].median():.2f}")

resolved_df = df[df["n_skills"] > 0]
print(f"\nAmong jobs WITH at least one skill (n={len(resolved_df)}):")
print(f"Mean number of skills per job:   {resolved_df['n_skills'].mean():.2f}")
print(f"Median number of skills per job: {resolved_df['n_skills'].median():.2f}")

print("\nMean skill count by country and type:")
print(df.groupby(["country", "type"])["n_skills"].mean().round(2).unstack(fill_value=0))

Total records loaded: 219809

SAMPLE: first 5 records

--- Record 1 ---
{
  "organization": "ENVISTONE, spol. s r.o.",
  "skills": [
    "http://data.europa.eu/esco/skill/1fe682ad-b8a2-44bb-a557-7eccf79b2014",
    "http://data.europa.eu/esco/skill/5cddc62d-3958-4390-b17e-0aecb5848c40"
  ],
  "occupations": [
    "http://data.europa.eu/esco/occupation/fd6a1c18-b0ff-419f-a36a-e05957452d55",
    "http://data.europa.eu/esco/isco/C8189"
  ],
  "sectors": [
    "Collection of non-hazardous waste",
    "Other waste recovery",
    "Incineration without energy recovery",
    "Landfilling or permanent storage"
  ],
  "id": 6247696,
  "title": "OPERATING WASTE PROCESSING AND RECYCLING EQUIPMENT, Operating waste processing and recycling equipment (except metal waste)",
  "description": "CZ-ISCO: 81892 WASTE PROCESSING AND RECYCLING EQUIPMENT OPERATOR (except metal waste) WASTE PROCESSING AND RECYCLING EQUIPMENT OPERATOR - Place of work: Královéhradecky region - PP for an indefinite period, startin

#Merging Jobs And Profiles Datasets

In [10]:
# ============================================================
# MERGE jobs_all_filled.json AND profiles_all_filled.json
# INTO ONE COMBINED DATASET (with column unification & cleanup)
# ============================================================

import json
import pandas as pd

JOBS_PATH = "/content/jobs_all_filled.json"
PROFILES_PATH = "/content/profiles_all_filled.json"
OUTPUT_PATH = "/content/combined_all.json"

# ---- 1. LOAD BOTH DATASETS ----
with open(JOBS_PATH, encoding="utf-8") as f:
    jobs_df = pd.DataFrame(json.load(f))
with open(PROFILES_PATH, encoding="utf-8") as f:
    profiles_df = pd.DataFrame(json.load(f))

print(f"Jobs:     {len(jobs_df)} records, columns: {jobs_df.columns.tolist()}")
print(f"Profiles: {len(profiles_df)} records, columns: {profiles_df.columns.tolist()}")

# ---- 2. RENAME COLUMNS THAT MEAN THE SAME THING BUT HAVE DIFFERENT NAMES ----
# jobs.title / profiles.occupation      -> occupation_title
# jobs.occupations / profiles.occupation_uris -> occupation_uris
# jobs.organization / profiles.company  -> organization
jobs_df = jobs_df.rename(columns={
    "title": "occupation_title",
    "occupations": "occupation_uris",
    # "organization" already has the right name in jobs, no rename needed
})
profiles_df = profiles_df.rename(columns={
    "occupation": "occupation_title",
    "company": "organization",
    # "occupation_uris" already has the right name in profiles, no rename needed
})

# ---- 3. DROP COLUMNS THAT ARE REDUNDANT OR ALREADY DOCUMENTED AS UNRELIABLE ----
# Jobs: experience_level and nuts1/nuts2/nuts3 were already flagged in the
# Data Exploration Report as systematically missing / unusable for comparison
jobs_df = jobs_df.drop(columns=["experience_level", "nuts1", "nuts2", "nuts3"], errors="ignore")

# Profiles: "location" duplicates "country" (NOT the same meaning as jobs.location,
# which is a real city-level field) -> drop it to avoid a false merge.
# "user_country" / "user_location" are raw duplicates of "country" / "city".
profiles_df = profiles_df.drop(columns=["location", "user_country", "user_location"], errors="ignore")

# ---- 4. NORMALIZE "country" VALUES (jobs used lowercase filename slugs) ----
COUNTRY_SLUG_TO_NAME = {
    "germany": "Germany",
    "france": "France",
    "sweden": "Sweden",
    "romania": "Romania",
    "czechia": "Czech Republic",
    "poland": "Poland",
    "italy": "Italy",
    "spain": "Spain",
    "greece": "Greece",
}
jobs_df["country"] = jobs_df["country"].str.lower().map(COUNTRY_SLUG_TO_NAME).fillna(jobs_df["country"])
profiles_df["country"] = profiles_df["country"].str.strip()  # just in case of stray whitespace

# Sanity check: make sure both sides now use the exact same country labels
jobs_countries = set(jobs_df["country"].unique())
profiles_countries = set(profiles_df["country"].unique())
print(f"\nCountries only in jobs (not matched):     {jobs_countries - profiles_countries}")
print(f"Countries only in profiles (not matched):  {profiles_countries - jobs_countries}")

# ---- 5. MARK ENTITY TYPE (job vs profile) ----
jobs_df["entity_type"] = "job"
profiles_df["entity_type"] = "profile"

# ---- 6. BUILD A GLOBALLY UNIQUE KEY (id alone is only unique WITHIN each table) ----
jobs_df["original_id"] = jobs_df["id"]
profiles_df["original_id"] = profiles_df["id"]
jobs_df["record_key"] = "job_" + jobs_df["id"].astype(str)
profiles_df["record_key"] = "profile_" + profiles_df["id"].astype(str)
jobs_df = jobs_df.drop(columns=["id"])
profiles_df = profiles_df.drop(columns=["id"])

# ---- 7. CONCATENATE: union of all remaining columns, missing ones filled with NaN ----
combined_df = pd.concat([jobs_df, profiles_df], ignore_index=True, sort=False)

print(f"\nCombined: {len(combined_df)} records, {combined_df.shape[1]} columns")
print(f"Final columns: {combined_df.columns.tolist()}")

# ---- 8. SANITY CHECKS ----
assert combined_df["record_key"].is_unique, "Duplicate record_key found!"
print("\nrecord_key is unique across the combined dataset. ✔")

# ---- 9. SAVE ----
combined_df.to_json(OUTPUT_PATH, orient="records", force_ascii=False, indent=2)
print(f"\nSaved combined dataset to: {OUTPUT_PATH}")

Jobs:     219809 records, columns: ['organization', 'skills', 'occupations', 'sectors', 'id', 'title', 'description', 'experience_level', 'type', 'location', 'location_code', 'nuts1', 'nuts2', 'nuts3', 'upload_date', 'source', 'source_id', 'country', 'skills_source']
Profiles: 334151 records, columns: ['skills', 'sectors', 'id', 'full_name', 'location', 'content', 'occupation', 'occupation_uris', 'city', 'company', 'country', 'degree', 'description', 'ethnicity_predicted', 'highest_degree', 'region', 'sex_predicted', 'user_country', 'user_location', 'startdate', 'university_name', 'university_raw', 'university_country', 'university_location', 'ultimate_parent_school_name', 'url', 'source', 'source_id', 'type', 'skills_source']

Countries only in jobs (not matched):     set()
Countries only in profiles (not matched):  set()

Combined: 553960 records, 32 columns
Final columns: ['organization', 'skills', 'occupation_uris', 'sectors', 'occupation_title', 'description', 'type', 'location', 

In [12]:
# ============================================================
# INTEGRITY CHECK FOR combined_all.json
# Three levels: (1) encoding, (2) JSON structure, (3) data sanity
# ============================================================

import os
import json
import hashlib
import pandas as pd

COMBINED_PATH = "/content/combined_all.json"

print("=" * 60)
print("LEVEL 0: FILE BASICS")
print("=" * 60)
size_bytes = os.path.getsize(COMBINED_PATH)
print(f"File: {COMBINED_PATH}")
print(f"Size: {size_bytes:,} bytes ({size_bytes / (1024**2):.1f} MB)")

# Optional: checksum, useful later to confirm a copy (e.g. on Drive) is identical
sha256 = hashlib.sha256()
with open(COMBINED_PATH, "rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        sha256.update(chunk)
print(f"SHA256: {sha256.hexdigest()}")

print("\n" + "=" * 60)
print("LEVEL 1: ENCODING CHECK (can every byte be read as UTF-8?)")
print("=" * 60)
try:
    with open(COMBINED_PATH, "r", encoding="utf-8") as f:
        raw_text = f.read()
    print(f"OK - file fully decoded as UTF-8 ({len(raw_text):,} characters)")
    encoding_ok = True
except UnicodeDecodeError as e:
    print(f"FAILED at byte position {e.start}: {e}")
    print(f"  -> position is at {e.start / size_bytes:.1%} of the file")
    encoding_ok = False

# Peek at the very last characters - a healthy JSON array should end with "]"
if encoding_ok:
    tail = raw_text.rstrip()[-20:]
    print(f"Last 20 characters of file: {tail!r}")
    if not tail.endswith("]"):
        print("  [WARNING] File does not end with ']' - looks TRUNCATED, even though it decoded as UTF-8.")

print("\n" + "=" * 60)
print("LEVEL 2: JSON STRUCTURE CHECK (is it valid, complete JSON?)")
print("=" * 60)
if encoding_ok:
    try:
        data = json.loads(raw_text)
        print(f"OK - valid JSON, top-level type: {type(data).__name__}")
        json_ok = isinstance(data, list)
        if not json_ok:
            print("  [WARNING] Top-level JSON is not a list - unexpected structure.")
    except json.JSONDecodeError as e:
        print(f"FAILED: {e}")
        print(f"  -> line {e.lineno}, column {e.colno}, char position {e.pos}")
        print(f"  -> position is at {e.pos / len(raw_text):.1%} of the file")
        json_ok = False
        data = None
else:
    json_ok = False
    data = None

if json_ok:
    print("\n" + "=" * 60)
    print("LEVEL 3: DATA SANITY CHECKS")
    print("=" * 60)
    df = pd.DataFrame(data)
    print(f"Total records: {len(df)}")
    print(f"Columns ({len(df.columns)}): {df.columns.tolist()}")

    # 3a. record_key must be unique (this is the whole point of that column)
    if "record_key" in df.columns:
        n_unique = df["record_key"].nunique()
        n_total = len(df)
        if n_unique == n_total:
            print(f"record_key uniqueness: OK ({n_unique}/{n_total} unique)")
        else:
            print(f"[WARNING] record_key uniqueness FAILED: {n_unique}/{n_total} unique "
                  f"-> {n_total - n_unique} duplicates")
    else:
        print("[WARNING] 'record_key' column not found")

    # 3b. Critical columns should have zero missing values
    for col in ["entity_type", "country", "type", "record_key"]:
        if col in df.columns:
            n_missing = df[col].isna().sum()
            status = "OK" if n_missing == 0 else f"[WARNING] {n_missing} missing"
            print(f"'{col}' missing values: {status}")

    # 3c. Expected structure: only "job" and "profile" as entity_type
    if "entity_type" in df.columns:
        print(f"\nentity_type distribution:\n{df['entity_type'].value_counts()}")

    # 3d. Breakdown by entity_type / country / type - compare this against
    # the numbers you already saw when you built jobs_all_filled.json and
    # profiles_all_filled.json separately, to confirm nothing was lost or duplicated
    print("\nRecords per entity_type / country / type:")
    print(df.groupby(["entity_type", "country", "type"]).size().unstack(fill_value=0))

    # 3e. Check the first AND last record - corruption from truncation
    # typically shows up at the END, so checking only row 0 is not enough
    print("\nFirst record keys:", list(data[0].keys())[:5], "...")
    print("Last record keys: ", list(data[-1].keys())[:5], "...")

print("\n" + "=" * 60)
print("FINAL VERDICT")
print("=" * 60)
if encoding_ok and json_ok:
    print("File appears INTACT: readable as UTF-8, valid JSON, structurally consistent.")
else:
    print("File appears CORRUPTED - see the failed level above for where it broke.")

LEVEL 0: FILE BASICS
File: /content/combined_all.json
Size: 2,166,590,332 bytes (2066.2 MB)
SHA256: ef594b99238549e4d1fd45ed19e141be74e6a76e13055e242241d8eadccd8634

LEVEL 1: ENCODING CHECK (can every byte be read as UTF-8?)
OK - file fully decoded as UTF-8 (2,163,300,629 characters)
Last 20 characters of file: 'ist-82b08a314"\n  }\n]'

LEVEL 2: JSON STRUCTURE CHECK (is it valid, complete JSON?)
OK - valid JSON, top-level type: list

LEVEL 3: DATA SANITY CHECKS
Total records: 553960
Columns (32): ['organization', 'skills', 'occupation_uris', 'sectors', 'occupation_title', 'description', 'type', 'location', 'location_code', 'upload_date', 'source', 'source_id', 'country', 'skills_source', 'entity_type', 'original_id', 'record_key', 'full_name', 'content', 'city', 'degree', 'ethnicity_predicted', 'highest_degree', 'region', 'sex_predicted', 'startdate', 'university_name', 'university_raw', 'university_country', 'university_location', 'ultimate_parent_school_name', 'url']
record_key uniqu

In [16]:
# ============================================================
# INSPECTION + RENAME ON combined_all.json
# 1) Unique values of 'type' (green / lignite)
# 2) Unique values of 'entity_type' (job / profile)
# 3) Rename 'original_id' -> 'id'
# 4) Check 'skills' and 'occupation_uris' for missing values
# 5) Missing-value percentage for every other column
# ============================================================

import os
import json
import pandas as pd

COMBINED_PATH = "content/combined_all.json"
OUTPUT_PATH = "content/final_dataset.json"    # overwrite after rename

# ---- LOAD THE COMBINED DATASET ----
with open("combined_all.json", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)
print(f"Total records: {len(df)}")

# ---- 1. UNIQUE VALUES OF 'type' (should only be green / lignite) ----
print("\n" + "=" * 60)
print("UNIQUE VALUES: 'type'")
print("=" * 60)
print(df["type"].unique())

# ---- 2. UNIQUE VALUES OF 'entity_type' (should only be job / profile) ----
print("\n" + "=" * 60)
print("UNIQUE VALUES: 'entity_type'")
print("=" * 60)
print(df["entity_type"].unique())

# ---- 3. RENAME 'original_id' -> 'id' ----
df = df.rename(columns={"original_id": "id"})
print("\n" + "=" * 60)
print("RENAME CHECK: 'original_id' -> 'id'")
print("=" * 60)
print("'original_id' still present:", "original_id" in df.columns)
print("'id' now present:           ", "id" in df.columns)

# ---- 4. CHECK 'skills' AND 'occupation_uris' HAVE VALUES IN EVERY RECORD ----
# A record counts as "missing" here if the value is None, NaN, or an empty list
def is_missing_list_field(value):
    if value is None:
        return True
    if isinstance(value, float):  # NaN
        return True
    if isinstance(value, list) and len(value) == 0:
        return True
    return False

print("\n" + "=" * 60)
print("CHECK: 'skills' and 'occupation_uris' have values in all records")
print("=" * 60)
for col in ["skills", "occupation_uris"]:
    missing_mask = df[col].apply(is_missing_list_field)
    n_missing = missing_mask.sum()
    if n_missing == 0:
        print(f"'{col}': OK, all {len(df)} records have at least one value")
    else:
        print(f"'{col}': {n_missing} records ({n_missing/len(df):.2%}) are missing/empty")

# ---- 5. MISSING-VALUE PERCENTAGE FOR ALL OTHER COLUMNS ----
other_columns = [c for c in df.columns if c not in ["skills", "occupation_uris"]]

def is_missing_generic(value):
    if value is None:
        return True
    if isinstance(value, float):  # NaN
        return True
    if isinstance(value, list) and len(value) == 0:
        return True
    if isinstance(value, str) and value.strip() == "":
        return True
    return False

print("\n" + "=" * 60)
print("MISSING VALUE PERCENTAGE FOR ALL OTHER COLUMNS")
print("=" * 60)
missing_report = {}
for col in other_columns:
    n_missing = df[col].apply(is_missing_generic).sum()
    missing_report[col] = round(n_missing / len(df) * 100, 2)

missing_df = pd.Series(missing_report, name="missing_%").sort_values(ascending=False)
print(missing_df)

# ---- SAVE THE UPDATED DATASET (with 'id' column renamed) ----
df.to_json(OUTPUT_PATH, orient="records", force_ascii=False, indent=2)
print(f"\nSaved updated dataset (with 'id' renamed) to: {OUTPUT_PATH}")

Total records: 553960

UNIQUE VALUES: 'type'
['green' 'lignite']

UNIQUE VALUES: 'entity_type'
['job' 'profile']

RENAME CHECK: 'original_id' -> 'id'
'original_id' still present: False
'id' now present:            True

CHECK: 'skills' and 'occupation_uris' have values in all records
'skills': OK, all 553960 records have at least one value
'occupation_uris': 252 records (0.05%) are missing/empty

MISSING VALUE PERCENTAGE FOR ALL OTHER COLUMNS
university_location            94.59
content                        83.78
startdate                      74.05
ultimate_parent_school_name    69.90
degree                         69.89
university_name                69.89
university_country             69.89
university_raw                 69.89
upload_date                    69.48
location                       69.29
location_code                  60.32
description                    55.08
ethnicity_predicted            39.69
full_name                      39.68
sex_predicted                  39.6

OSError: Cannot save file into a non-existent directory: 'content'

In [1]:
# ============================================================
# CLEAN UP combined_all.json (Google Colab version)
# 1) Drop columns that are no longer needed
# 2) Rename 'original_id' -> 'id'
# 3) Save the result as final_dataset.json
# ============================================================

import json
import pandas as pd

COMBINED_PATH = "/content/combined_all.json"
OUTPUT_PATH = "/content/final_dataset.json"

# ---- 1. LOAD THE COMBINED DATASET ----
with open(COMBINED_PATH, encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)
print(f"Loaded {len(df)} records, {df.shape[1]} columns")
print(f"Columns before cleanup: {df.columns.tolist()}")

# ---- 2. DROP THE REQUESTED COLUMNS ----
columns_to_drop = [
    "university_location",
    "content",
    "startdate",
    "ultimate_parent_school_name",
    "degree",
    "university_name",
    "university_country",
    "university_raw",
    "upload_date",
    "location",
    "location_code",
]

# errors="ignore" makes this safe even if a column is already missing
df = df.drop(columns=columns_to_drop, errors="ignore")

# ---- 3. RENAME 'original_id' -> 'id' ----
df = df.rename(columns={"original_id": "id"})

print(f"\nColumns after cleanup: {df.columns.tolist()}")
print(f"Total columns now: {df.shape[1]}")
print(f"'id' present: {'id' in df.columns}")
print(f"'original_id' present: {'original_id' in df.columns}")

# ---- 4. SAVE AS final_dataset.json ----
df.to_json(OUTPUT_PATH, orient="records", force_ascii=False, indent=2)
print(f"\nSaved cleaned dataset to: {OUTPUT_PATH}")
print(f"Final shape: {df.shape[0]} records, {df.shape[1]} columns")

Loaded 553960 records, 32 columns
Columns before cleanup: ['organization', 'skills', 'occupation_uris', 'sectors', 'occupation_title', 'description', 'type', 'location', 'location_code', 'upload_date', 'source', 'source_id', 'country', 'skills_source', 'entity_type', 'original_id', 'record_key', 'full_name', 'content', 'city', 'degree', 'ethnicity_predicted', 'highest_degree', 'region', 'sex_predicted', 'startdate', 'university_name', 'university_raw', 'university_country', 'university_location', 'ultimate_parent_school_name', 'url']

Columns after cleanup: ['organization', 'skills', 'occupation_uris', 'sectors', 'occupation_title', 'description', 'type', 'source', 'source_id', 'country', 'skills_source', 'entity_type', 'id', 'record_key', 'full_name', 'city', 'ethnicity_predicted', 'highest_degree', 'region', 'sex_predicted', 'url']
Total columns now: 21
'id' present: True
'original_id' present: False

Saved cleaned dataset to: /content/final_dataset.json
Final shape: 553960 records, 